# 6장 — 시험의 오차범위 📏

여론조사의 그 문장("1,004명, 95% 신뢰수준, ±3.1%p")과 벤치마크 점수의 오차범위는
**같은 공식**입니다:

$$\hat{p} \;\pm\; 1.96 \times \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$

- p̂(읽기: **피-햇**. 모자 ^ 는 "표본에서 **추정한** 값"이라는 통계 관례) = 표본 정답률
- 표준오차 √(p̂(1−p̂)/n): 표본 정답률의 출렁임 한 단위 (3부 7장의 √n)
- ±1.96: 출렁임의 95%가 담기는 범위 (3부 6장 정규분포의 95% 규칙)

3부 8장(추정·신뢰구간)을 드디어 실전에 쓰는 장입니다.

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 여기서 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch6')

## 1. 오차범위 계산기: 냅킨 공식 98/√n 검증

In [ ]:
import math

def half_width(p, n):
    """95% 신뢰구간의 반폭 (비율 단위)."""
    return 1.96 * math.sqrt(p * (1 - p) / n)

print('   n    | ±폭 (p̂=50%) | 냅킨 98/√n')
print('-' * 40)
for n in [50, 100, 250, 1000, 5000, 10000]:
    hw = half_width(0.5, n) * 100
    print(f'{n:>7} | {hw:9.2f}%p | {98/math.sqrt(n):7.2f}%p')

assert round(half_width(0.5, 1000) * 100, 1) == 3.1, '여론조사의 ±3.1%p'
assert round(half_width(0.5, 100) * 100, 1) == 9.8
assert round(half_width(0.5, 10000) * 100, 2) == 0.98
assert round(half_width(0.7, 1000) * 100, 1) == 2.8, 'p̂=70%는 최악값보다 좁다'
assert round(half_width(0.8, 164) * 100, 1) == 6.1, '164문항 벤치마크의 ±'
print()
print('✔ n=1,000·p̂=50% → ±3.1%p. 여론조사와 벤치마크는 같은 수학')

## 2. "95%"의 진짜 뜻: 참값 담기 100회 실험 (신의 시점)

참 실력 p=70%를 정해 두고 시험을 100번 봅니다. 매번 95% 구간을 만들면 몇 번 참값을 담을까요?
난수는 웹 랩과 같은 mulberry32(시드 600+회차)라 숫자가 정확히 일치합니다.

In [ ]:
M32 = 0xFFFFFFFF

def imul(a, b):
    return (a * b) & M32

def mulberry32(seed):
    a = seed & M32
    def rand():
        nonlocal a
        a = (a + 0x6D2B79F5) & M32
        t = a
        t = imul(t ^ (t >> 15), t | 1) & M32
        t = (t ^ ((t + imul(t ^ (t >> 7), t | 61)) & M32)) & M32
        return (t ^ (t >> 14)) / 4294967296
    return rand

P_TRUE = 0.7

def coverage(n, rounds=100):
    miss = 0
    intervals = []
    for r in range(1, rounds + 1):
        rand = mulberry32(600 + r)
        c = sum(1 for _ in range(n) if rand() < P_TRUE)
        phat = c / n
        h = half_width(phat, n)
        missed = P_TRUE < phat - h or P_TRUE > phat + h
        miss += missed
        intervals.append((phat - h, phat + h, missed))
    return miss, intervals

miss100, iv100 = coverage(100)
miss1000, iv1000 = coverage(1000)
print(f'100문항 시험 100회: 참값(70%)을 놓친 구간 {miss100}회')
print(f'1,000문항 시험 100회: 놓친 구간 {miss1000}회   (이론값 ≈ 5회)')
assert miss100 == 2 and miss1000 == 7, '웹 랩과 동일 — 놓침 횟수도 5회 근처에서 출렁이는 표본'
print()
print('✔ 95% = 이 절차의 장기 명중률. 특정 구간 하나가 맞을 확률이 아니다')

## 3. 그림: 100개의 구간과 참값

In [ ]:
import matplotlib.pyplot as plt

try:
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['axes.unicode_minus'] = False
except Exception:
    pass

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, (iv, n, miss) in zip(axes, [(iv100, 100, miss100), (iv1000, 1000, miss1000)]):
    for i, (lo, hi, missed) in enumerate(iv):
        ax.plot([i, i], [lo * 100, hi * 100], color='#e11d48' if missed else '#a5b4fc',
                lw=2 if missed else 1)
    ax.axhline(70, color='#f59e0b', ls='--', lw=1.5)
    ax.set_title(f'{n}문항 × 100회: 놓침 {miss}회 (빨강)')
    ax.set_xlabel('시험 회차')
axes[0].set_ylabel('점수 구간 (%)')
plt.tight_layout()
plt.show()
print('구간 폭 자체도 다르다는 것에 주목. 1,000문항 쪽이 훨씬 좁다(√n의 힘)')

## 4. 리더보드에 밴드 씌우기

가상 리더보드(아라-2 72.4 / 바다-3 71.8 / 사라-1 69.0 / 다라-7 65.2)에
문항 수별 신뢰구간을 씌우고 인접 순위의 겹침을 판정합니다.

In [ ]:
LB = [('아라-2', 0.724), ('바다-3', 0.718), ('사라-1', 0.690), ('다라-7', 0.652)]

def separated(a, b, n):
    return a - half_width(a, n) > b + half_width(b, n)

print('   n    | 1↔2위 | 2↔3위 | 3↔4위')
print('-' * 40)
results = {}
for n in [250, 1000, 5000, 10000]:
    marks = [separated(LB[i][1], LB[i + 1][1], n) for i in range(3)]
    results[n] = marks
    print(f'{n:>7} | ' + ' | '.join('분리 ' if m else '겹침 ' for m in marks))

assert results[1000] == [False, False, False], '1,000문항: 순위 전체 판정 불가'
assert results[5000] == [False, True, True]
assert results[10000] == [False, True, True], '0.6점 차 1·2위는 10,000문항으로도 못 가른다'
print()
print('✔ 밴드를 씌우는 순간 리더보드의 절반은 "판정 보류". 정식 판정 도구는 7장(짝지은 비교)')

## 정리: 오늘 가져갈 세 문장

1. 벤치마크 오차범위 = **p̂ ± 1.96·√(p̂(1−p̂)/n)**. 여론조사와 같은 수학(3부 8장 복습).
2. **95%는 절차의 명중률**: 100번 만들면 약 95번 참값을 담는 절차라는 뜻.
3. 냅킨 공식 **± ≈ 98/√n %p**. 점수를 보면 문항 수부터 확인하는 습관.

다음 **7장(차이가 진짜일까)**: 겹친 두 모델을 가리는 정식 도구, 짝지은 비교와 p값(3부 9장 복습).